# HW3 Pipeline Results Analysis
Loads `results/metrics.jsonl` and visualises latency, token usage, and cost per pipeline step.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

METRICS_PATH = Path('../results/metrics.jsonl')
rows = [json.loads(line) for line in METRICS_PATH.read_text().splitlines() if line.strip()]
df = pd.DataFrame(rows)
df['ts'] = pd.to_datetime(df['ts'])
print(f'Loaded {len(df)} metric records spanning {df["model"].nunique()} model(s)')
df.head()

In [ ]:
# Summary statistics
summary = df.agg(
    total_calls=('step', 'count'),
    total_input_tokens=('input_tokens', 'sum'),
    total_output_tokens=('output_tokens', 'sum'),
    total_cost_usd=('cost_usd', 'sum'),
    mean_latency_ms=('latency_ms', 'mean'),
    max_latency_ms=('latency_ms', 'max'),
)
print(summary.to_string())

In [ ]:
# Identify broad step categories
def categorise(step: str) -> str:
    if step.startswith('label:'):
        return 'label'
    if step.startswith('judge'):
        return 'judge'
    if step.startswith('review'):
        return 'review'
    if 'draft' in step or 'writing' in step:
        return 'writing'
    if 'guideline' in step:
        return 'guideline'
    return 'other'

df['category'] = df['step'].apply(categorise)
cat_counts = df['category'].value_counts()
print('Step categories:\n', cat_counts.to_string())

In [ ]:
# Plot 1 — Latency distribution per category
fig, ax = plt.subplots(figsize=(9, 4))
categories = df['category'].unique()
data_by_cat = [df.loc[df['category'] == c, 'latency_ms'].values for c in categories]
ax.boxplot(data_by_cat, labels=categories, vert=True)
ax.set_title('Latency Distribution per Step Category')
ax.set_ylabel('Latency (ms)')
ax.set_xlabel('Category')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2 — Token usage over time (input vs output)
df_sorted = df.sort_values('ts')
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(range(len(df_sorted)), df_sorted['input_tokens'], alpha=0.5, label='Input tokens')
ax.fill_between(range(len(df_sorted)), df_sorted['output_tokens'], alpha=0.5, label='Output tokens')
ax.set_title('Token Usage per LLM Call (chronological)')
ax.set_xlabel('Call index')
ax.set_ylabel('Tokens')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3 — Cumulative cost over time
df_sorted = df.sort_values('ts').copy()
df_sorted['cumulative_cost'] = df_sorted['cost_usd'].cumsum()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sorted['ts'], df_sorted['cumulative_cost'], linewidth=2)
ax.set_title('Cumulative Pipeline Cost (USD) Over Time')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Cumulative Cost (USD)')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4 — Cost per category (pie chart)
cost_by_cat = df.groupby('category')['cost_usd'].sum()
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(cost_by_cat.values, labels=cost_by_cat.index, autopct='%1.1f%%', startangle=140)
ax.set_title(f'Total Cost by Category  (total: ${cost_by_cat.sum():.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 5 — Model comparison: mean latency and mean cost
model_stats = df.groupby('model').agg(
    mean_latency=('latency_ms', 'mean'),
    mean_cost=('cost_usd', 'mean'),
    call_count=('step', 'count'),
).reset_index()
print(model_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(model_stats['model'], model_stats['mean_latency'])
axes[0].set_title('Mean Latency per Model (ms)')
axes[0].set_xlabel('ms')
axes[1].barh(model_stats['model'], model_stats['mean_cost'])
axes[1].set_title('Mean Cost per Call per Model (USD)')
axes[1].set_xlabel('USD')
for ax in axes:
    ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 6 — Top 10 most expensive individual steps
top_steps = df.nlargest(10, 'cost_usd')[['step', 'model', 'latency_ms', 'cost_usd']]
print('Top 10 most expensive calls:')
print(top_steps.to_string(index=False))

In [ ]:
# Plot 7 — Latency histogram
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['latency_ms'], bins=40, edgecolor='white', alpha=0.8)
ax.axvline(df['latency_ms'].median(), color='red', linestyle='--', label=f'Median {df["latency_ms"].median():.0f} ms')
ax.set_title('Latency Distribution Across All LLM Calls')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print('=== Pipeline Cost Summary ===')
print(f'Total LLM calls  : {len(df):,}')
print(f'Total input tok  : {df["input_tokens"].sum():,}')
print(f'Total output tok : {df["output_tokens"].sum():,}')
print(f'Total cost (USD) : ${df["cost_usd"].sum():.4f}')
print(f'Mean latency     : {df["latency_ms"].mean():.0f} ms')
print(f'Max latency      : {df["latency_ms"].max():.0f} ms')
print(f'Models used      : {", ".join(df["model"].unique())}')